# IEEE-CIS fraud lifecycle on Kaggle

## Goal
Run the repository's validated CLI against attached IEEE-CIS files. This notebook contains orchestration only; core data, feature, model, monitoring, and diagnosis logic remains in `fraud_monitor`.

## Setup

Set `RUN_FULL_PIPELINE=True` in Kaggle. If the repository is not already present under `/kaggle/working`, also provide its public Git URL. Internet access is required only for cloning/installing; the pipeline itself is file based.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

RUN_FULL_PIPELINE = False
REPOSITORY_URL = os.environ.get("FRAUD_MONITOR_REPOSITORY", "")
PROJECT_ROOT = (
    Path("/kaggle/working/financial-fraud-detection") if Path("/kaggle").exists() else Path.cwd()
)
WORK_ROOT = (
    Path("/kaggle/working/fraud-monitor-run")
    if Path("/kaggle").exists()
    else PROJECT_ROOT / "artifacts" / "private" / "notebook-smoke"
)
print(
    {
        "project_root": str(PROJECT_ROOT),
        "work_root": str(WORK_ROOT),
        "run_full_pipeline": RUN_FULL_PIPELINE,
    }
)

In [ ]:
if RUN_FULL_PIPELINE and not (PROJECT_ROOT / "pyproject.toml").is_file():
    if not REPOSITORY_URL:
        raise RuntimeError("Set FRAUD_MONITOR_REPOSITORY to the public Git repository URL.")
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
if RUN_FULL_PIPELINE:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", f"{PROJECT_ROOT}[train]"], check=True
    )
else:
    print("Dry run: repository installation skipped.")

## Steps

Discover the single attached competition directory, then execute preparation, training, replay, and public aggregate export in order.

In [ ]:
raw_dir = None
if RUN_FULL_PIPELINE:
    candidates = sorted(Path("/kaggle/input").glob("**/train_transaction.csv"))
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one train_transaction.csv, found {len(candidates)}.")
    raw_dir = candidates[0].parent
    required = {
        "train_transaction.csv",
        "train_identity.csv",
        "test_transaction.csv",
        "test_identity.csv",
    }
    missing = required - {path.name for path in raw_dir.iterdir()}
    if missing:
        raise RuntimeError(f"Attached dataset is missing: {sorted(missing)}")
    print(f"Using competition files from {raw_dir}")
else:
    print("Dry run: Kaggle input discovery skipped.")

In [ ]:
def run_cli(*arguments: str) -> None:
    command = [sys.executable, "-m", "fraud_monitor.cli", *arguments]
    print("Running:", " ".join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)


processed_dir = WORK_ROOT / "processed"
model_dir = WORK_ROOT / "model"
monitoring_dir = WORK_ROOT / "monitoring"
demo_dir = WORK_ROOT / "demo"

if RUN_FULL_PIPELINE:
    run_cli("prepare", "--raw-dir", str(raw_dir), "--output-dir", str(processed_dir))
    run_cli("train", "--processed-dir", str(processed_dir), "--output-dir", str(model_dir))
    run_cli(
        "replay",
        "--processed-dir",
        str(processed_dir),
        "--bundle",
        str(model_dir / "model_bundle.joblib"),
        "--output-dir",
        str(monitoring_dir),
    )
    run_cli(
        "build-demo",
        "--source-dir",
        str(monitoring_dir),
        "--output-dir",
        str(demo_dir),
        "--review-budget",
        str(model_dir / "acceptance_review_budgets.parquet"),
    )
else:
    print("Dry run: CLI execution skipped. Set RUN_FULL_PIPELINE=True on Kaggle.")

## Checks

Inspect bounded aggregate summaries. Do not display or export transaction rows.

In [ ]:
if RUN_FULL_PIPELINE:
    training_summary = json.loads((model_dir / "training_summary.json").read_text())
    monitoring_manifest = json.loads((monitoring_dir / "monitoring_manifest.json").read_text())
    print(
        json.dumps(
            {
                "model_version": training_summary["model_version"],
                "temporal_cv_pr_auc": training_summary["tuning"]["mean_pr_auc"],
                "acceptance_pr_auc": training_summary["acceptance_metrics"]["pr_auc"],
                "logistic_pr_auc": training_summary["baseline_pr_auc"]["logistic"],
                "paired_improvement_interval": training_summary["acceptance_intervals"][
                    "pr_auc_improvement_over_logistic"
                ],
                "production_batches": monitoring_manifest["production_batches"],
                "shadow_batches": monitoring_manifest["shadow_batches"],
            },
            indent=2,
        )
    )
else:
    print("Dry run complete: notebook structure is valid; no model claims were produced.")

## Next steps

Download private run artifacts for review. Commit only the allow-listed aggregate demo export after confirming that it contains no row identifiers or competition data. Trigger `retrain-eval` manually only when monitoring evidence warrants a challenger.